In [ ]:
import pandas as pd

# 1. Load the Embeddings

gvp_df = pd.read_parquet("../../Data/3. Protein_enbeddings/GVP-GNN_protein_embeddings.parquet")
esm_df = pd.read_parquet("../../Data/3. Protein_enbeddings/ESM_embeddings_(t33_650m model).parquet")
egnn_df = pd.read_parquet("../../Data/2. Drug_embeddings/EGNN_drug_embeddings_v2.parquet")
chem_df = pd.read_parquet("../../Data/2. Drug_embeddings/smiles_embeddings_chemberta.parquet")

# 2. Load your main dataset (Replace with your actual path)
# This file must have the columns that link to the embeddings (e.g., 'protein_id' and 'drug_id')
main_df = pd.read_parquet("../../Data/scope_onside_common_v3.parquet")
adr_df = pd.read_parquet("../../Data/final_rxnorm_meddra_v2.parquet")

# 3. Perform Merges
# Note: Adjust 'on=' to match the ID column names in your specific files (e.g., 'SMILES' or 'UniProtID')

# Merge Protein Embeddings
combined_df = main_df.merge(gvp_df, on='protein_id', how='left')
combined_df = combined_df.merge(esm_df, on='protein_id', how='left')

# Merge Drug Embeddings
combined_df = combined_df.merge(egnn_df, on='drug_id', how='left')
combined_df = combined_df.merge(chem_df, on='drug_id', how='left')

# 4. Handle Missing Values
# It is vital to drop rows where embeddings are missing to avoid NaN errors in Torch
combined_df = combined_df.dropna(subset=[
    'esm_embedding', 'gvp_embedding', 
    'chemberta_embedding', 'egnn_embedding'
])

# 5. Save the Final Dataset
final_path = "../../Data/processed_dti_dataset.parquet"
combined_df.to_parquet(final_path, index=False)

print(f"Final dataset saved with {len(combined_df)} rows.")

FileNotFoundError: [Errno 2] No such file or directory: '../../Data/1. Raw/interaction_data.parquet'

In [5]:
import pandas as pd

# Define paths
paths = {
    "gvp": "../../Data/3. Protein_enbeddings/GVP-GNN_protein_embeddings.parquet",
    "esm": "../../Data/3. Protein_enbeddings/ESM_embeddings_(t33_650m model).parquet",
    "egnn": "../../Data/2. Drug_embeddings/EGNN_drug_embeddings_v2.parquet",
    "chem": "../../Data/2. Drug_embeddings/smiles_embeddings_chemberta.parquet",
    "main": "../../Data/scope_onside_common_v3.parquet",
    "adr": "../../Data/final_rxnorm_meddra_v2.parquet"
}

# Load dataframes into a dictionary
dfs = {}
for name, path in paths.items():
    try:
        dfs[name] = pd.read_parquet(path)
        print(f"✅ Loaded {name}: {dfs[name].shape}")
    except Exception as e:
        print(f"❌ Error loading {name}: {e}")

# Display column names for inspection
print("\n--- Column Names ---")
for name, df in dfs.items():
    print(f"{name.upper()}: {df.columns.tolist()}")

✅ Loaded gvp: (2385, 8)
✅ Loaded esm: (2385, 4)
✅ Loaded egnn: (1028, 3)
✅ Loaded chem: (1028, 5)
✅ Loaded main: (34741, 7)
✅ Loaded adr: (69474, 3)

--- Column Names ---
GVP: ['uniprot_id', 'length', 'mean_pLDDT', 'embedding_dim', 'encoder_version', 'pdb_md5', 'embedding', 'source']
ESM: ['id', 'length', 'dim', 'embedding']
EGNN: ['drug_chembl_id', 'rxcui', 'embedding']
CHEM: ['drug_chembl_id', 'smiles', 'embedding', 'embedding_dim', 'model_name']
MAIN: ['drug_chembl_id', 'target_uniprot_id', 'label', 'smiles', 'sequence', 'molfile_3d', 'rxcui']
ADR: ['rxnorm_ingredient_id', 'meddra_id', 'meddra_name']


In [6]:
import pandas as pd
import numpy as np

# 1. Prepare Protein Embeddings
# Rename 'id' in ESM to 'uniprot_id' and 'embedding' to 'esm_embedding'
esm_prep = dfs['esm'][['id', 'embedding']].rename(columns={'id': 'target_uniprot_id', 'embedding': 'esm_embedding'})
gvp_prep = dfs['gvp'][['uniprot_id', 'embedding']].rename(columns={'uniprot_id': 'target_uniprot_id', 'embedding': 'gvp_embedding'})

# 2. Prepare Drug Embeddings
# Rename 'embedding' to distinguish between models
egnn_prep = dfs['egnn'][['drug_chembl_id', 'embedding']].rename(columns={'embedding': 'egnn_embedding'})
chem_prep = dfs['chem'][['drug_chembl_id', 'embedding']].rename(columns={'embedding': 'chemberta_embedding'})

# 3. Prepare ADR IDs (Group them into a list per drug)
# We map them via 'rxcui' which exists in both MAIN and (presumably) corresponds to rxnorm_id
adr_grouped = dfs['adr'].groupby('rxnorm_ingredient_id')['meddra_id'].apply(list).reset_index()
adr_grouped.columns = ['rxcui', 'adr_ids']

# 4. Start Merging on the MAIN dataframe
print("Merging dataframes...")
final_df = dfs['main'].copy()

# Merge Proteins
final_df = final_df.merge(esm_prep, on='target_uniprot_id', how='left')
final_df = final_df.merge(gvp_prep, on='target_uniprot_id', how='left')

# Merge Drugs
final_df = final_df.merge(egnn_prep, on='drug_chembl_id', how='left')
final_df = final_df.merge(chem_prep, on='drug_chembl_id', how='left')

# Merge ADR list
final_df = final_df.merge(adr_grouped, on='rxcui', how='left')

# 5. Final Cleanup
# Drop rows where we are missing any of the 4 core embeddings
initial_count = len(final_df)
final_df = final_df.dropna(subset=['esm_embedding', 'gvp_embedding', 'egnn_embedding', 'chemberta_embedding'])
print(f"Rows after dropping missing embeddings: {len(final_df)} (Dropped {initial_count - len(final_df)})")

# Fill empty ADR lists with empty list instead of NaN
final_df['adr_ids'] = final_df['adr_ids'].apply(lambda d: d if isinstance(d, list) else [])

# 6. Save for the Dataset class
output_path = "../../Data/processed_dti_dataset.parquet"
final_df.to_parquet(output_path, index=False)
print(f"Successfully saved to {output_path}")

Merging dataframes...
Rows after dropping missing embeddings: 34741 (Dropped 0)
Successfully saved to ../../Data/processed_dti_dataset.parquet
